#  Curso: Langfuse

Este notebook instrumenta `cloudbox_bot()` y **envía las trazas a la nube de Langfuse**.
No se quedan en el notebook: al terminar, las verás en la web de tu proyecto.

### ¿Cómo funciona? (el flujo de datos)
1. Instrumentamos el sistema con `observe(...)` (spans y una *generation*).
2. El SDK de Langfuse acumula los spans en memoria mientras corre el código.
3. **`langfuse.flush()`** los envía por HTTPS a `https://us.cloud.langfuse.com`.
4. Abres tu proyecto en la web → **Tracing / Traces** → y ahí está cada ejecución,
   con su árbol, tokens, costo y latencia.

> **¿Veré algo en la web?** Sí. Con claves válidas, las trazas van a la nube y las
> exploras en la UI. (Sin claves, un notebook solo imprimiría el árbol localmente —
> aquí no, porque vamos a usar tus claves.)


### 🔐 Nota de seguridad (importante)
La `LANGFUSE_SECRET_KEY` (`sk-lf-...`) es una **credencial**. Buenas prácticas:
- **Rótala** en *Settings → API Keys* si la compartiste en algún sitio (chat, captura, repo).
- **No repartas** a los alumnos un notebook con la secret key incrustada: que cada uno ponga
  la suya, o usa *Colab Secrets* (`google.colab.userdata`) como se muestra en comentarios abajo.


## 1. Instalación

In [ ]:
!pip install transformers torch huggingface_hub pandas rank_bm25 -q
!pip install langfuse -q   # SDK v4 (OpenTelemetry). Requiere Python 3.11+

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


## 2. Acceso a Gemma
Acepta la licencia en https://huggingface.co/google/gemma-3-1b-it y crea un token *Read* en
https://huggingface.co/settings/tokens. Si ya hiciste login en esta sesión, salta esta celda.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Cargar el LLM y utilidades

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time, uuid
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("🔄 Loading LLM...")
model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

def ask_llm(prompt, max_new_tokens=60):
    out = llm([{"role": "user", "content": prompt}],
              max_new_tokens=max_new_tokens, do_sample=False,
              pad_token_id=tokenizer.eos_token_id, return_full_text=False)
    return out[0]["generated_text"].strip()

def count_tokens(text):
    return len(tokenizer.encode(text))

print("✅ LLM ready")

🔄 Loading LLM...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✅ LLM ready


## 4. El sistema a observar: `cloudbox_bot` (retriever BM25 + generador)

In [ ]:
CORPUS = [
    {"id": "doc-01", "text": "CloudBox ofrece 15 GB de almacenamiento gratuito en el plan Free."},
    {"id": "doc-02", "text": "El plan Pro de CloudBox cuesta 9 dolares al mes e incluye 2 TB."},
    {"id": "doc-03", "text": "Para restaurar un archivo borrado, ve a la Papelera; se guardan 30 dias."},
    {"id": "doc-04", "text": "CloudBox cifra los archivos en reposo con AES-256 y en transito con TLS."},
    {"id": "doc-05", "text": "Puedes compartir una carpeta con un enlace de solo lectura o edicion."},
    {"id": "doc-06", "text": "El limite por archivo es 50 GB en Pro y 5 GB en Free."},
    {"id": "doc-07", "text": "CloudBox sincroniza en Windows, macOS, Android e iOS."},
    {"id": "doc-08", "text": "El soporte 24/7 por chat es solo para clientes del plan Pro."},
]
_bm25 = BM25Okapi([d["text"].lower().split() for d in CORPUS])

def retriever(query, k=3):
    scores = _bm25.get_scores(query.lower().split())
    return [CORPUS[i] for i in np.argsort(scores)[::-1][:k]]

def build_prompt(query, docs):
    contexto = "\n".join(f"- {d['text']}" for d in docs)
    return ("Responde usando SOLO el contexto. Si no esta, di que no lo sabes.\n\n"
            f"Contexto:\n{contexto}\n\nPregunta: {query}\nRespuesta:")

print("✅ Sistema listo")

✅ Sistema listo


## 5. Conectar con Langfuse  ⬅️ *aquí van tus claves*

Si `auth_check()` devuelve `True`, estás conectado y todo lo que tracees aparecerá en la web.

In [ ]:
import os

# ============ TUS CLAVES (rota la secret key cuando termines de probar) ============
os.environ["LANGFUSE_HOST"]       = "https://us.cloud.langfuse.com"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-8c53a053-7a99-4afe-9d78-ffc5ecc10432"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-ef93aaf8-901b-4c6b-83e2-ce51c55d9f1c"
# ===================================================================================
#
# 🔒 MEJOR PRÁCTICA (para repartir a alumnos) — usa Colab Secrets en vez de lo de arriba:
#     import os
#     os.environ["LANGFUSE_HOST"]       = "https://us.cloud.langfuse.com"
#     os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
#     os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
#   (defines los secretos en el panel 🔑 de la izquierda de Colab)

from langfuse import get_client
langfuse = get_client()

if langfuse.auth_check():
    print("✅ Conectado a Langfuse:", os.environ["LANGFUSE_HOST"])
else:
    print("❌ auth_check fallo. Revisa host, public key y secret key.")

✅ Conectado a Langfuse: https://us.cloud.langfuse.com


## 6. Instrumentar `cloudbox_bot`

Dos tipos de observación:
- **span** para pasos normales (`retrieve`).
- **generation** para la llamada al LLM: Langfuse trata modelo, tokens y costo de forma nativa.

Estructura del trace:
```
cloudbox_bot            (span raíz)
├── retrieve            (span)
└── generate ⚡         (generation)  -> model, prompt, respuesta, tokens, costo
```

In [ ]:
from langfuse import propagate_attributes  # patron v4 para user_id/session_id

# Precios de ejemplo (USD por 1K tokens), como si Gemma fuera una API de pago
PRICE_IN, PRICE_OUT = 0.0005, 0.0015

def cloudbox_bot_traced(query, session_id="clase6-demo", user_id="user-0"):
    # span raiz -> su input/output se convierten en el input/output de la traza
    with langfuse.start_as_current_observation(as_type="span", name="cloudbox_bot") as root:
        root.update(input={"query": query})

        # propagate_attributes: aplica session_id/user_id a esta y todas las observaciones hijas
        with propagate_attributes(session_id=session_id, user_id=user_id,
                                  trace_name="cloudbox_bot"):

            # 1) Retrieval -> span
            with langfuse.start_as_current_observation(as_type="span", name="retrieve") as r:
                docs = retriever(query)
                r.update(output={"doc_ids": [d["id"] for d in docs]}, metadata={"k": len(docs)})

            # 2) Generación -> observación tipo generation
            prompt = build_prompt(query, docs)
            with langfuse.start_as_current_observation(
                    as_type="generation", name="generate", model="gemma-3-1b-it") as gen:
                gen.update(input=prompt)
                answer = ask_llm(prompt)
                n_in, n_out = count_tokens(prompt), count_tokens(answer)
                cost = n_in/1000*PRICE_IN + n_out/1000*PRICE_OUT
                gen.update(
                    output=answer,
                    usage_details={"input": n_in, "output": n_out, "total": n_in + n_out},
                    cost_details={"input": n_in/1000*PRICE_IN,
                                  "output": n_out/1000*PRICE_OUT, "total": cost},
                )

        root.update(output={"answer": answer})
        return answer

print("✅ cloudbox_bot instrumentado")

✅ cloudbox_bot instrumentado


## 7. Enviar tu primera traza y verla en la web

In [ ]:
ans = cloudbox_bot_traced("¿Cuanto cuesta el plan Pro y cuanto espacio da?")
langfuse.flush()

print("Respuesta:", ans)
print("\n➡️  Abre https://us.cloud.langfuse.com  ->  tu proyecto  ->  Tracing")
print("   Deberias ver un trace 'cloudbox_bot' con retrieve + generate.")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: El plan Pro cuesta 9 dolares al mes e incluye 2 TB.

➡️  Abre https://us.cloud.langfuse.com  ->  tu proyecto  ->  Tracing
   Deberias ver un trace 'cloudbox_bot' con retrieve + generate.


## 8. Simular tráfico (para que la UI tenga con qué trabajar)

In [ ]:
queries = [
    "¿Como restauro un archivo borrado?",
    "¿CloudBox cifra mis archivos?",
    "¿Cual es el limite de tamano por archivo?",
    "¿En que plataformas sincroniza?",
    "¿Cual es la capital de Francia?",   # fuera del corpus: buen caso para observar
]
for i, q in enumerate(queries):
    cloudbox_bot_traced(q, session_id="clase6-demo", user_id=f"user-{i%3}")

langfuse.flush()
print(f"✅ {len(queries)} trazas mas enviadas. Refresca la UI de Langfuse.")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

✅ 5 trazas mas enviadas. Refresca la UI de Langfuse.


## 9. Qué mirar en la web de Langfuse

En `https://us.cloud.langfuse.com`, dentro de tu proyecto:

- **Tracing / Traces** — la lista de ejecuciones. Entra a una: verás el árbol
  `cloudbox_bot → retrieve → generate` con la duración de cada paso.
- **El span `generate`** — muestra el prompt, la respuesta, y **tokens + costo** (porque lo
  marcamos como *generation*). Ahí ves la factura estimada por llamada.
- **Sessions** — agrupadas por `session_id` (`clase6-demo`). Filtra por `user_id`.
- **Dashboards** — costo total, latencia, nº de trazas en el tiempo.
- **Agent graph view** — cuando instrumentes tu grafo LangGraph, aquí verás los nodos como
  grafo y los bucles (tu retry loop) como ciclos. Anticipo de la Clase 8.

Si **no ves nada**: 99% de las veces es (a) olvidar `langfuse.flush()`, o (b) `auth_check()`
en `False` (claves/host mal). No es que "se quede en el notebook".


## 10. (Opcional) Correr Langfuse en local con Docker

Todo lo de arriba es **idéntico** apuntando a una instancia local: solo cambian los 3 valores
de la celda 5. Así lo levantas en tu máquina (nada sale a internet):

In [ ]:
# ==========================================================================
#  LANGFUSE AUTOHOSPEDADO CON DOCKER  (ejecutar en tu terminal, NO en Colab)
# ==========================================================================
#
#  Colab no puede levantar el stack (son varios contenedores). Esto es para
#  Jupyter local o un servidor del curso.
#
#  1) Clona el repo y levanta los servicios:
#       git clone https://github.com/langfuse/langfuse.git
#       cd langfuse
#       docker compose up -d
#
#  2) Abre la UI en tu navegador:
#       http://localhost:3000
#     Crea una cuenta local -> crea una organizacion -> crea un proyecto.
#
#  3) En el proyecto: Settings -> API Keys -> crea un par de claves.
#
#  4) En la celda 5 de ESTE notebook (o en tu Jupyter local), cambia SOLO el host
#     y pega las claves LOCALES:
#
#       os.environ["LANGFUSE_HOST"]       = "http://localhost:3000"
#       os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-...(local)"
#       os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-...(local)"
#
#  5) Corre el resto igual. Las trazas ahora viven en TU maquina.
#
#  Nota: 'docker compose up' arranca Langfuse + Postgres + Clickhouse + Redis +
#  almacenamiento. Requiere Docker instalado y unos ~4 GB de RAM libres.
# ==========================================================================
print("Guia Docker: ver comentarios de esta celda.")

Guia Docker: ver comentarios de esta celda.
